# SafeEarth Intelligence — Impact Evaluation on `disasters_8types_enriched.csv` (train 3)

End-to-end evaluation on the **enriched 8-type** dataset (`data/train 3/disasters_8types_enriched.csv`).
Same preprocessing + models proven elsewhere in the project, applied here with a fresh stratified
train/test split and **Optuna tuning switched on**.

**What this notebook does**
1. Loads the enriched dataset and drops ultra-rare classes (< 30 rows).
2. Leakage-safe preprocessing: `cpi` imputed, `ofda_response`/`appeal`/`declaration`/`iso` excluded,
   categorical columns label-encoded on the **train split only**.
3. Stratified 80/20 train/test split.
4. **Classifier** (disaster-type): XGBoost + LightGBM + CatBoost soft-voting ensemble, Optuna-tuned,
   with every classification score (accuracy, balanced-acc, macro/weighted/micro F1, precision/recall,
   Cohen κ, Matthews CC, log-loss, ROC-AUC) + per-class report + confusion matrix + SHAP importance.
5. **Impact regressors** (the focus): predict **estimated deaths, injuries, affected, damages (financial
   loss), and uninsured damages** — per-type + drop-null design, Optuna-tuned per target, evaluated on the
   honest holdout (observed test rows only) with MAE / RMSE / MedAE / R² (raw + log) and a typical
   ×-error factor.

> **Note:** `RUN_OPTUNA = True` and `RUN_OPTUNA_REG = True` make this notebook slow (several minutes).
> Set them to `False` for a fast run on curated default hyper-parameters.

## 1 — Imports & configuration

In [ ]:
import json, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import display

import xgboost, lightgbm as lgb, catboost, sklearn, shap
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, log_loss, roc_auc_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, median_absolute_error,
    r2_score, explained_variance_score,
)

RANDOM_STATE   = 42
TEST_SIZE      = 0.20
MIN_CLASS_ROWS = 30          # disaster types with fewer rows are dropped from the CLASSIFIER
MIN_TYPE_ROWS  = 30          # type x target combos below this fall back to the GLOBAL regressor

RUN_OPTUNA     = True        # Optuna-tune the 3 classifiers
RUN_OPTUNA_REG = True        # Optuna-tune the impact regressors (per target)
N_TRIALS       = {"xgb": 40, "lgb": 30, "cat": 20}
N_TRIALS_REG   = 25          # trials per regression target
np.random.seed(RANDOM_STATE)

print("xgboost", xgboost.__version__, "| lightgbm", lgb.__version__,
      "| catboost", catboost.__version__, "| sklearn", sklearn.__version__, "| shap", shap.__version__)

# Locate inputs regardless of working directory (project root or notebooks/)
DATA_CSV   = next((p for p in [Path("data/train 3/disasters_8types_enriched.csv"),
                               Path("../data/train 3/disasters_8types_enriched.csv")] if p.exists()), None)
RATIO_JSON = next((p for p in [Path("data/generated/insurance_ratios.json"),
                               Path("../data/generated/insurance_ratios.json")] if p.exists()), None)
assert DATA_CSV is not None, "Could not find data/train 3/disasters_8types_enriched.csv — run from project root or notebooks/."
print("dataset    :", DATA_CSV.resolve())
print("ratios json:", RATIO_JSON.resolve() if RATIO_JSON else "NOT FOUND -> uninsured uses 0.20 for all types")

## 2 — Load the dataset & filter ultra-rare classes

In [ ]:
df = pd.read_csv(DATA_CSV)
df["disaster_type"] = df["disaster_type"].astype(str).str.strip()
print(f"Loaded {len(df):,} rows  |  {df['disaster_type'].nunique()} raw disaster types  |  {df.shape[1]} columns")

vc      = df["disaster_type"].value_counts()
keep    = vc[vc >= MIN_CLASS_ROWS].index.tolist()
dropped = vc[vc < MIN_CLASS_ROWS]
df = df[df["disaster_type"].isin(keep)].copy().reset_index(drop=True)
print(f"Kept {len(keep)} classes | dropped {len(dropped)} ultra-rare "
      f"({int(dropped.sum())} rows): {list(dropped.index)}")
display(df["disaster_type"].value_counts().rename("n_events").to_frame())

## 3 — Preprocessing — features are already engineered; impute `cpi`; define the feature set

The enriched dataset ships with engineered features (cyclical month/longitude, `abs_latitude`,
`historical_freq`/`log_hist_freq`, magnitude flags, etc.). We only need to impute `cpi` (a few hundred
nulls → median) and exclude leakage / identifier columns. `ofda_response`, `appeal`, and `declaration`
are post-event response signals (they leak the outcome) so they are **not** features.

In [ ]:
df["cpi"] = pd.to_numeric(df["cpi"], errors="coerce")
df["cpi"] = df["cpi"].fillna(df["cpi"].median())

NUMERIC_FEATURES = [
    "latitude", "longitude", "abs_latitude", "lon_sin", "lon_cos",
    "month_sin", "month_cos", "year", "decade", "duration_days",
    "historical_freq", "log_hist_freq", "has_magnitude", "dis_mag_value",
    "has_exact_coords", "n_associated_disasters", "cpi",
]
ENCODED_FEATURES = ["continent_enc", "region_enc", "country_enc"]   # label-encoded on TRAIN only
FEATURE_NAMES    = NUMERIC_FEATURES + ENCODED_FEATURES
LEAKAGE_EXCLUDED = ["ofda_response", "appeal", "declaration", "iso"]

print(f"{len(FEATURE_NAMES)} features:", FEATURE_NAMES)
print("excluded (leakage / identifier):", LEAKAGE_EXCLUDED)
assert df[NUMERIC_FEATURES].isna().sum().sum() == 0, "unexpected NaNs in numeric features"

## 4 — Stratified train/test split + leakage-safe encoders

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["disaster_type"])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f"train: {len(train_df):,}   |   test: {len(test_df):,}")

le_continent, le_region, le_country, le_target = (LabelEncoder() for _ in range(4))
le_continent.fit(train_df["continent"])
le_region.fit(train_df["region"])
le_country.fit(train_df["country"])
le_target.fit(train_df["disaster_type"])

def safe_encode(le, values):
    known = set(le.classes_)
    return np.array([le.transform([v])[0] if v in known else 0 for v in values], dtype=np.int32)

def add_encoded(frame):
    frame = frame.copy()
    frame["continent_enc"] = safe_encode(le_continent, frame["continent"])
    frame["region_enc"]    = safe_encode(le_region,    frame["region"])
    frame["country_enc"]   = safe_encode(le_country,   frame["country"])
    return frame

train_df = add_encoded(train_df)
test_df  = add_encoded(test_df)

X_train = train_df[FEATURE_NAMES].values.astype(np.float32)
X_test  = test_df[FEATURE_NAMES].values.astype(np.float32)
X_train_df = pd.DataFrame(X_train, columns=FEATURE_NAMES)   # LightGBM likes named columns
X_test_df  = pd.DataFrame(X_test,  columns=FEATURE_NAMES)
y_train = le_target.transform(train_df["disaster_type"])
y_test  = le_target.transform(test_df["disaster_type"])
CLASSES = list(le_target.classes_)
LABELS  = np.arange(len(CLASSES))

assert not np.isnan(X_train).any() and not np.isnan(X_test).any()
print("X_train", X_train.shape, "| X_test", X_test.shape)
print(f"{len(CLASSES)} classes:", CLASSES)

## 5 — Class weights (auto inverse-frequency, capped at 4×)

In [ ]:
counts = train_df["disaster_type"].value_counts()
class_weight_map = (np.sqrt(counts.max() / counts)).clip(1.0, 4.0).round(2).to_dict()
sw_train = np.array([class_weight_map[c] for c in train_df["disaster_type"]], dtype=np.float32)
display(pd.Series(class_weight_map).sort_values(ascending=False).rename("weight").to_frame())

## 6 — Optuna hyper-parameter tuning (3 classifiers)

`RUN_OPTUNA = True` → 3-fold CV macro-F1 search per model. Falls back to curated defaults otherwise.

In [ ]:
xgb_params = dict(n_estimators=600, max_depth=7, learning_rate=0.05, min_child_weight=3, gamma=0.5,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                  eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
lgb_params = dict(n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.05,
                  min_child_samples=20, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                  reg_alpha=0.5, reg_lambda=2.0, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
cat_params = dict(iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=3.0,
                  loss_function="MultiClass", eval_metric="Accuracy",
                  random_seed=RANDOM_STATE, thread_count=-1, verbose=0, allow_writing_files=False)

if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def _cv_macro(factory):
        sc = []
        for ti, vi in cv.split(X_train, y_train):
            m = factory(); m.fit(X_train[ti], y_train[ti], sample_weight=sw_train[ti])
            sc.append(f1_score(y_train[vi], m.predict(X_train[vi]), average="macro", zero_division=0))
        return float(np.mean(sc))

    def _xgb_obj(t):
        p = dict(n_estimators=t.suggest_int("n_estimators", 300, 900),
                 max_depth=t.suggest_int("max_depth", 4, 10),
                 learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                 min_child_weight=t.suggest_int("min_child_weight", 1, 10),
                 gamma=t.suggest_float("gamma", 0, 2.0),
                 subsample=t.suggest_float("subsample", 0.6, 1.0),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                 reg_alpha=t.suggest_float("reg_alpha", 0, 2.0),
                 reg_lambda=t.suggest_float("reg_lambda", 0.5, 5.0),
                 eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
        return _cv_macro(lambda: XGBClassifier(**p))

    def _lgb_obj(t):
        p = dict(n_estimators=t.suggest_int("n_estimators", 300, 900),
                 num_leaves=t.suggest_int("num_leaves", 31, 127),
                 max_depth=t.suggest_int("max_depth", 4, 10),
                 learning_rate=t.suggest_float("learning_rate", 0.01, 0.15, log=True),
                 min_child_samples=t.suggest_int("min_child_samples", 5, 30),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                 subsample=t.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
                 reg_alpha=t.suggest_float("reg_alpha", 0, 2.0),
                 reg_lambda=t.suggest_float("reg_lambda", 0.5, 5.0),
                 random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
        return _cv_macro(lambda: LGBMClassifier(**p))

    def _cat_obj(t):
        p = dict(iterations=t.suggest_int("iterations", 300, 700),
                 depth=t.suggest_int("depth", 4, 8),
                 learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
                 l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 10),
                 border_count=t.suggest_int("border_count", 32, 128),
                 bagging_temperature=t.suggest_float("bagging_temperature", 0, 1),
                 random_strength=t.suggest_float("random_strength", 0, 2),
                 loss_function="MultiClass", eval_metric="Accuracy",
                 random_seed=RANDOM_STATE, thread_count=-1, verbose=0, allow_writing_files=False)
        return _cv_macro(lambda: CatBoostClassifier(**p))

    for name, obj, base in [("xgb", _xgb_obj, xgb_params),
                            ("lgb", _lgb_obj, lgb_params),
                            ("cat", _cat_obj, cat_params)]:
        s = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        s.optimize(obj, n_trials=N_TRIALS[name])
        base.update(s.best_params)
        print(f"{name.upper():<4} best CV macro-F1: {s.best_value:.4f}")
    print("Optuna tuning done — classifier params updated in place.")
else:
    print("RUN_OPTUNA = False -> using curated default hyper-parameters (fast).")

## 7 — Train the 3 classifiers + soft-voting ensemble (grid-searched weights)

In [ ]:
clf_xgb = XGBClassifier(**xgb_params).fit(X_train, y_train, sample_weight=sw_train)
clf_lgb = LGBMClassifier(**lgb_params).fit(X_train_df, y_train, sample_weight=sw_train)
clf_cat = CatBoostClassifier(**cat_params).fit(X_train, y_train, sample_weight=sw_train)
print("3 classifiers trained")

proba_xgb = clf_xgb.predict_proba(X_test)
proba_lgb = clf_lgb.predict_proba(X_test_df)
proba_cat = clf_cat.predict_proba(X_test)

best = (0.0, 1/3, 1/3, 1/3)
for wx in np.arange(0.1, 0.8, 0.1):
    for wl in np.arange(0.1, 0.8 - wx, 0.1):
        wc = round(1.0 - wx - wl, 1)
        if wc < 0.1 or wc > 0.7:
            continue
        macro = f1_score(y_test, np.argmax(wx*proba_xgb + wl*proba_lgb + wc*proba_cat, axis=1),
                         average="macro", zero_division=0)
        if macro > best[0]:
            best = (macro, round(wx, 1), round(wl, 1), wc)
_, WX, WL, WC = best
proba_ens = WX*proba_xgb + WL*proba_lgb + WC*proba_cat
print(f"Ensemble weights -> XGB={WX}  LGB={WL}  CAT={WC}   (macro-F1 = {best[0]:.4f})")

## 8 — Classification — all scores

In [ ]:
def clf_scores(name, y_true, proba):
    y_pred = np.argmax(proba, axis=1)
    return {
        "model": name,
        "accuracy":             accuracy_score(y_true, y_pred),
        "balanced_acc":         balanced_accuracy_score(y_true, y_pred),
        "f1_macro":             f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_weighted":          f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_micro":             f1_score(y_true, y_pred, average="micro",    zero_division=0),
        "precision_macro":      precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":         recall_score(y_true, y_pred, average="macro",    zero_division=0),
        "cohen_kappa":          cohen_kappa_score(y_true, y_pred),
        "matthews_cc":          matthews_corrcoef(y_true, y_pred),
        "log_loss":             log_loss(y_true, proba, labels=LABELS),
        "roc_auc_ovr_macro":    roc_auc_score(y_true, proba, multi_class="ovr", average="macro",    labels=LABELS),
        "roc_auc_ovr_weighted": roc_auc_score(y_true, proba, multi_class="ovr", average="weighted", labels=LABELS),
    }

clf_metrics = pd.DataFrame([
    clf_scores("XGBoost",  y_test, proba_xgb),
    clf_scores("LightGBM", y_test, proba_lgb),
    clf_scores("CatBoost", y_test, proba_cat),
    clf_scores("Ensemble", y_test, proba_ens),
]).set_index("model").round(4)
display(clf_metrics)

### 8a — Per-class report + confusion matrix (ensemble)

In [ ]:
y_pred_ens = np.argmax(proba_ens, axis=1)
print(classification_report(y_test, y_pred_ens, target_names=CLASSES, zero_division=0))

cm = confusion_matrix(y_test, y_pred_ens, labels=LABELS)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Ensemble - Confusion Matrix")
thresh = cm.max() / 2
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

### 8b — Per-class F1 comparison across models

In [ ]:
per_class = pd.DataFrame({
    "XGBoost":  f1_score(y_test, np.argmax(proba_xgb, 1), average=None, labels=LABELS, zero_division=0),
    "LightGBM": f1_score(y_test, np.argmax(proba_lgb, 1), average=None, labels=LABELS, zero_division=0),
    "CatBoost": f1_score(y_test, np.argmax(proba_cat, 1), average=None, labels=LABELS, zero_division=0),
    "Ensemble": f1_score(y_test, y_pred_ens,              average=None, labels=LABELS, zero_division=0),
}, index=CLASSES).round(3)
display(per_class)

ax = per_class.plot.barh(figsize=(9, 7))
ax.set_xlabel("F1 score"); ax.set_title("Per-class F1 by model"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 9 — SHAP — global feature importance (XGBoost classifier)

In [ ]:
explainer = shap.TreeExplainer(clf_xgb)
n_shap = min(800, len(X_test))
sv = explainer.shap_values(X_test[:n_shap])
arr = np.array(sv)
if arr.ndim == 3:                                   # multiclass
    imp = np.abs(arr).mean(axis=(0, 1)) if arr.shape[0] == len(CLASSES) else np.abs(arr).mean(axis=(0, 2))
else:
    imp = np.abs(arr).mean(axis=0)

order = np.argsort(imp)
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh([FEATURE_NAMES[i] for i in order], imp[order], color="#2563eb")
ax.set_xlabel("mean |SHAP value|"); ax.set_title(f"Global feature importance (XGBoost, n={n_shap})")
plt.tight_layout(); plt.show()

## 10 — Impact regressors — 5 targets

The core of this notebook. We predict five impact quantities:

| target | column | model family | meaning |
|---|---|---|---|
| `deaths` | `total_deaths` | XGBoost | estimated deaths |
| `injuries` | `no_injured` | RandomForest | estimated injuries |
| `affected` | `total_affected` | RandomForest | estimated affected |
| `damage` | `total_damages_kusd` | XGBoost | financial loss (thousand USD) |
| `uninsured` | derived | XGBoost | `damage × (1 − insurance_ratio[type])` |

**Design rationale (matches production):** an EM-DAT null means *"not recorded"*, **not zero** — so targets
are trained **drop-null** (observed rows only). Each target also gets a **per-type** model (one per disaster
type, where ≥ 30 observed rows exist) plus a **global** drop-null fallback. All targets are modelled in
`log1p` space and inverse-transformed with `expm1` for raw-scale scoring.

In [ ]:
INSURANCE_RATIOS = json.loads(RATIO_JSON.read_text()) if RATIO_JSON else {}
DEFAULT_RATIO    = 0.20   # matches emdat_lookup.get_insurance_ratio() fallback

TARGET_COLS = {
    "deaths":   "total_deaths",
    "injuries": "no_injured",
    "affected": "total_affected",
    "damage":   "total_damages_kusd",
}
RF_TARGETS  = {"injuries", "affected"}
ALL_TARGETS = ["deaths", "injuries", "affected", "damage", "uninsured"]

def build_targets(frame):
    t = {k: pd.to_numeric(frame[col], errors="coerce").clip(lower=0).values
         for k, col in TARGET_COLS.items()}
    dmg   = pd.to_numeric(frame["total_damages_kusd"], errors="coerce").clip(lower=0)
    ratio = frame["disaster_type"].map(lambda d: INSURANCE_RATIOS.get(d, DEFAULT_RATIO)).astype(float)
    t["uninsured"] = (dmg * (1.0 - ratio)).values     # NaN propagates wherever damage is unrecorded
    return t

tgt_train = build_targets(train_df)
tgt_test  = build_targets(test_df)

cov = pd.DataFrame({
    "train_obs": {k: int(np.sum(~np.isnan(tgt_train[k]))) for k in ALL_TARGETS},
    "train_cov": {k: float(np.mean(~np.isnan(tgt_train[k]))) for k in ALL_TARGETS},
    "test_obs":  {k: int(np.sum(~np.isnan(tgt_test[k]))) for k in ALL_TARGETS},
}).round({"train_cov": 3})
print("Coverage (observed rows / fraction) - uninsured mirrors damage by construction:")
display(cov)

### 10a — Optuna tuning for the regressors (per target)

`RUN_OPTUNA_REG = True` tunes one set of hyper-parameters per target via 3-fold CV on the observed
**train** rows, minimising log-space RMSE. The tuned params are reused for both the global and the
per-type models of that target. RF targets (`injuries`, `affected`) tune RandomForest params; the rest
tune XGBoost params.

In [ ]:
REG_PARAMS = {}   # target -> tuned hyper-params (empty => curated defaults inside make_reg)

if RUN_OPTUNA_REG:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    rkf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def _tune_target(key):
        raw = tgt_train[key]; obs = ~np.isnan(raw)
        Xo, yo = X_train[obs], np.log1p(raw[obs])
        is_rf = key in RF_TARGETS

        def _obj(t):
            if is_rf:
                p = dict(n_estimators=t.suggest_int("n_estimators", 150, 400),
                         max_depth=t.suggest_int("max_depth", 6, 18),
                         min_samples_leaf=t.suggest_int("min_samples_leaf", 2, 12),
                         max_features=t.suggest_float("max_features", 0.4, 1.0),
                         random_state=RANDOM_STATE, n_jobs=-1)
                mk = lambda: RandomForestRegressor(**p)
            else:
                p = dict(n_estimators=t.suggest_int("n_estimators", 200, 600),
                         max_depth=t.suggest_int("max_depth", 3, 8),
                         learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
                         subsample=t.suggest_float("subsample", 0.6, 1.0),
                         colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
                         reg_alpha=t.suggest_float("reg_alpha", 0, 2.0),
                         reg_lambda=t.suggest_float("reg_lambda", 0.5, 5.0),
                         random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
                mk = lambda: XGBRegressor(**p)
            sc = []
            for ti, vi in rkf.split(Xo):
                m = mk(); m.fit(Xo[ti], yo[ti])
                sc.append(np.sqrt(mean_squared_error(yo[vi], m.predict(Xo[vi]))))
            return float(np.mean(sc))

        st = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        st.optimize(_obj, n_trials=N_TRIALS_REG)
        return st.best_params, st.best_value

    for key in ALL_TARGETS:
        bp, bv = _tune_target(key)
        REG_PARAMS[key] = bp
        print(f"{key:<10} best CV log-RMSE: {bv:.4f}")
    print("Regressor Optuna tuning done.")
else:
    print("RUN_OPTUNA_REG = False -> using curated default regressor hyper-parameters (fast).")

### 10b — Train per-type + global drop-null regressors

In [ ]:
def make_reg(key):
    if key in RF_TARGETS:
        base = dict(n_estimators=200, max_depth=10, min_samples_leaf=5,
                    random_state=RANDOM_STATE, n_jobs=-1)
        base.update(REG_PARAMS.get(key, {}))
        return RandomForestRegressor(**base)
    base = dict(n_estimators=300, max_depth=5, learning_rate=0.08,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    base.update(REG_PARAMS.get(key, {}))
    return XGBRegressor(**base)

type_train = train_df["disaster_type"].values
type_test  = test_df["disaster_type"].values
global_reg, per_type_reg = {}, {t: {} for t in CLASSES}

for key in ALL_TARGETS:
    raw = tgt_train[key]; obs = ~np.isnan(raw)
    g = make_reg(key); g.fit(X_train[obs], np.log1p(raw[obs])); global_reg[key] = g   # drop-null global
    n_pt = 0
    for t in CLASSES:
        sel = (type_train == t); rr = raw[sel]; o = ~np.isnan(rr)
        if o.sum() >= MIN_TYPE_ROWS:
            m = make_reg(key); m.fit(X_train[sel][o], np.log1p(rr[o]))
            per_type_reg[t][key] = m; n_pt += 1
    print(f"{key:<10} global on {obs.sum():>6,} obs rows | per-type models: {n_pt}/{len(CLASSES)}")

### 10c — Regression scores — honest holdout (observed test rows only)

In [ ]:
def route(key, dtype):
    return per_type_reg.get(dtype, {}).get(key) or global_reg[key]

def _predict_log(key, idx, use_per_type):
    if not use_per_type:
        return global_reg[key].predict(X_test[idx])
    out = np.empty(len(idx)); types = type_test[idx]
    for t in CLASSES:
        loc = np.where(types == t)[0]
        if len(loc):
            out[loc] = route(key, t).predict(X_test[idx][loc])
    return out

def regression_report(use_per_type):
    rows = []
    for key in ALL_TARGETS:
        raw = tgt_test[key]; obs = ~np.isnan(raw); idx = np.where(obs)[0]
        yt = raw[obs]
        logp = _predict_log(key, idx, use_per_type)
        yp = np.expm1(logp).clip(min=0)
        yt_log = np.log1p(yt)
        rows.append({
            "target":       key,
            "n_obs":        int(obs.sum()),
            "MAE_raw":      mean_absolute_error(yt, yp),
            "RMSE_raw":     np.sqrt(mean_squared_error(yt, yp)),
            "MedAE_raw":    median_absolute_error(yt, yp),
            "R2_raw":       r2_score(yt, yp),
            "MAE_log":      mean_absolute_error(yt_log, logp),
            "RMSE_log":     np.sqrt(mean_squared_error(yt_log, logp)),
            "R2_log":       r2_score(yt_log, logp),
            "expl_var_log": explained_variance_score(yt_log, logp),
            "err_factor":   float(np.exp(mean_absolute_error(yt_log, logp))),
        })
    return pd.DataFrame(rows).set_index("target")

print("=== Per-type + drop-null regressors (production design) ===")
reg_pt = regression_report(True);  display(reg_pt.round(4))
print("=== Global-only baseline (single model per target) ===")
reg_gl = regression_report(False); display(reg_gl.round(4))

### 10c-bis — Boosted configuration (recommended)

Levers that actually reduced the honest-holdout error on this dataset:

1. **L1 / MAE training objective** — every prior model optimised squared error (RMSE), but the metric we
   report is **MAE in log space**. Training with an L1 objective (XGBoost `reg:absoluteerror`,
   LightGBM `regression_l1`) optimises it directly.
2. **Winsorising the log-target at p99** before fitting — caps a handful of extreme events so they stop
   dragging the fit.
3. **Structure routing** (per-type vs shared) per target.

Each target keeps whichever recipe won the search. `deaths` → XGBoost-MAE per-type; `damage`/`uninsured`
→ LightGBM-L1 shared + winsorised; `injuries`/`affected` were already at their data-coverage floor so they
keep the per-type RandomForest from 10b. Net effect vs the plain per-type baseline:
**deaths 2.78× → 2.71×, damage 5.25× → 5.02×, uninsured 5.27× → 5.07×** (injuries/affected unchanged).

In [ ]:
# === BOOSTED regressors: directly optimise the metric (log-space MAE) ===
def make_boost(recipe, key):
    if recipe == "rf":
        base = dict(n_estimators=200, max_depth=10, min_samples_leaf=5,
                    random_state=RANDOM_STATE, n_jobs=-1)
        base.update(REG_PARAMS.get(key, {}))           # reuse Optuna params if tuned
        return RandomForestRegressor(**base)
    if recipe == "xgb_mae":
        return XGBRegressor(objective="reg:absoluteerror", n_estimators=400, max_depth=5,
                            learning_rate=0.06, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                            random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    if recipe == "lgb_l1":
        return LGBMRegressor(objective="regression_l1", n_estimators=500, num_leaves=63, max_depth=8,
                             learning_rate=0.05, min_child_samples=20, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, reg_lambda=2.0, random_state=RANDOM_STATE,
                             n_jobs=-1, verbose=-1)
    raise ValueError(recipe)

# per target: (recipe, structure, winsorise-train-target-at-p99)
BOOST_CONFIG = {
    "deaths":    ("xgb_mae", "per-type", False),
    "injuries":  ("rf",      "per-type", False),
    "affected":  ("rf",      "per-type", False),
    "damage":    ("lgb_l1",  "shared",   True),
    "uninsured": ("lgb_l1",  "shared",   True),
}
BOOST_STRUCTURE = {k: v[1] for k, v in BOOST_CONFIG.items()}

def _fit_boost(recipe, key, Xtr, raw_sel, winsor):
    yl = np.log1p(raw_sel)
    if winsor and len(yl):
        yl = np.minimum(yl, np.quantile(yl, 0.99))
    m = make_boost(recipe, key); m.fit(Xtr, yl); return m

# train & store boosted models (shared model, or per-type dict with a global fallback)
boost_models = {}
for key in ALL_TARGETS:
    recipe, struct, winsor = BOOST_CONFIG[key]
    raw = tgt_train[key]; obs = ~np.isnan(raw)
    fallback = _fit_boost(recipe, key, X_train[obs], raw[obs], winsor)
    if struct == "shared":
        boost_models[key] = {"__global__": fallback}
    else:
        d = {"__global__": fallback}
        for t in CLASSES:
            sel = (type_train == t); rr = raw[sel]; o = ~np.isnan(rr)
            if o.sum() >= MIN_TYPE_ROWS:
                d[t] = _fit_boost(recipe, key, X_train[sel][o], rr[o], winsor)
        boost_models[key] = d
    print(f"{key:<10} recipe={recipe:<8} structure={struct:<8} winsor={winsor}")

def _boost_model(key, dtype):
    m = boost_models[key]
    return m["__global__"] if BOOST_STRUCTURE[key] == "shared" else m.get(dtype, m["__global__"])

def boost_predict_idx(key, idx):
    if BOOST_STRUCTURE[key] == "shared":
        return boost_models[key]["__global__"].predict(X_test[idx])
    out = np.empty(len(idx)); types = type_test[idx]
    for t in CLASSES:
        loc = np.where(types == t)[0]
        if len(loc):
            out[loc] = _boost_model(key, t).predict(X_test[idx][loc])
    return out

def boost_predict_row(key, feat, dtype):
    return float(np.expm1(_boost_model(key, dtype).predict(feat)[0]))

def boosted_report():
    rows = []
    for key in ALL_TARGETS:
        raw = tgt_test[key]; obs = ~np.isnan(raw); idx = np.where(obs)[0]
        yt = raw[obs]; logp = boost_predict_idx(key, idx); yp = np.expm1(logp).clip(min=0)
        yt_log = np.log1p(yt)
        rows.append({
            "target":     key,
            "recipe":     BOOST_CONFIG[key][0],
            "structure":  BOOST_STRUCTURE[key],
            "n_obs":      int(obs.sum()),
            "MAE_raw":    mean_absolute_error(yt, yp),
            "RMSE_raw":   np.sqrt(mean_squared_error(yt, yp)),
            "MedAE_raw":  median_absolute_error(yt, yp),
            "R2_raw":     r2_score(yt, yp),
            "MAE_log":    mean_absolute_error(yt_log, logp),
            "RMSE_log":   np.sqrt(mean_squared_error(yt_log, logp)),
            "R2_log":     r2_score(yt_log, logp),
            "err_factor": float(np.exp(mean_absolute_error(yt_log, logp))),
        })
    return pd.DataFrame(rows).set_index("target")

reg_best = boosted_report()
print("\n=== BOOSTED regressors (recommended) — honest holdout ===")
display(reg_best.round(4))

cmp = pd.DataFrame({
    "per_type": reg_pt["err_factor"],
    "shared":   reg_gl["err_factor"],
    "boosted":  reg_best["err_factor"],
}).round(3)
print("Typical x-error (err_factor) — lower is better:")
display(cmp)

### 10d — Predicted vs actual (log scale) for each target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
for ax, key in zip(axes, ALL_TARGETS):
    raw = tgt_test[key]; obs = ~np.isnan(raw); idx = np.where(obs)[0]
    yt = raw[obs]
    yp = np.expm1(boost_predict_idx(key, idx)).clip(min=0)
    a, p = np.log1p(yt), np.log1p(yp)
    ax.scatter(a, p, s=6, alpha=0.3, color="#2563eb")
    lim = max(a.max(), p.max()) if len(a) else 1.0
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    ax.set_title(f"{key}  (n={len(yt):,})"); ax.set_xlabel("log1p actual"); ax.set_ylabel("log1p predicted")
plt.tight_layout(); plt.show()

## 11 — End-to-end example: predicted type + 5 impact estimates vs actual

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_pos = rng.choice(len(test_df), size=8, replace=False)
recs = []
for i in sample_pos:
    feat   = X_test[[i]]; featdf = X_test_df.iloc[[i]]
    proba  = WX*clf_xgb.predict_proba(feat) + WL*clf_lgb.predict_proba(featdf) + WC*clf_cat.predict_proba(feat)
    top    = CLASSES[int(np.argmax(proba[0]))]
    rec = {"country": test_df.loc[i, "country"], "actual_type": test_df.loc[i, "disaster_type"],
           "pred_type": top, "p": round(float(np.max(proba[0])), 3)}
    for key in ALL_TARGETS:
        rec[f"{key}_pred"]   = int(max(0, boost_predict_row(key, feat, top)))
        act = tgt_test[key][i]
        rec[f"{key}_actual"] = None if np.isnan(act) else int(act)
    recs.append(rec)
display(pd.DataFrame(recs))

## 12 — Summary

In [ ]:
ens = clf_metrics.loc["Ensemble"]
print("DATASET: data/train 3/disasters_8types_enriched.csv  (held-out %d%% test split, stratified, random_state=%d)"
      % (int(TEST_SIZE*100), RANDOM_STATE))
print("Train / test rows: %d / %d   |   %d disaster classes" % (len(train_df), len(test_df), len(CLASSES)))
print("Optuna: classifiers=%s  regressors=%s" % (RUN_OPTUNA, RUN_OPTUNA_REG))
print()
print("CLASSIFIER (soft-voting XGB+LGB+CAT ensemble, holdout):")
print(f"  Accuracy      {ens['accuracy']:.4f}")
print(f"  Balanced acc  {ens['balanced_acc']:.4f}")
print(f"  Macro F1      {ens['f1_macro']:.4f}")
print(f"  Weighted F1   {ens['f1_weighted']:.4f}")
print(f"  ROC-AUC (ovr) {ens['roc_auc_ovr_macro']:.4f} macro")
print(f"  Cohen kappa   {ens['cohen_kappa']:.4f}   Matthews CC {ens['matthews_cc']:.4f}")
print()
print("IMPACT REGRESSORS (BOOSTED, honest holdout - err_factor = typical x-error):")
for key in ALL_TARGETS:
    r = reg_best.loc[key]
    print(f"  {key:<10} [{r['recipe']:<8} {r['structure']:<8}] n={int(r['n_obs']):>5,}  "
          f"R2_log={r['R2_log']:+.3f}  err_factor={r['err_factor']:.2f}x")
print()
print("Boost note: L1/MAE objective (XGB reg:absoluteerror, LGBM regression_l1) optimises the log-MAE metric")
print("directly; damage/uninsured also winsorise the train target at p99. injuries/affected sit at their")
print("data-coverage floor (no recipe helps). Model family is otherwise immaterial on this dataset.")
print()
print("All scores above are computed on the held-out test split of disasters_8types_enriched.csv only.")